# 정답지 -- Agent Service Deployment Basic 킬러 과제

> **용도**: 채점자/강사 전용 (학생 배포 금지)  
> **총점**: 100점 | **문제 수**: 9문제 (25개 서브문제)  
> **기준 코드베이스**: MetaCodeLecture-AgentServiceDeploymentBasic (section7 브랜치)

---

## 문제 1: Docker 기초 및 Dockerfile 작성 (12점)

### 1-1. Backend Dockerfile 직접 작성 (5점)

**채점 기준:**

| 항목 | 배점 | 기준 |
|------|------|------|
| `FROM python:3.11-slim` | 1점 | 정확한 베이스 이미지 |
| `RUN apt-get ... curl` | 0.5점 | curl 설치 + 캐시 정리 |
| `COPY pyproject.toml uv.lock ./` (소스보다 먼저) | 1점 | 레이어 캐싱 순서 정확 |
| `RUN pip install uv && uv sync` | 0.5점 | uv 설치 및 동기화 |
| `EXPOSE 8000` + `CMD` | 0.5점 | 올바른 포트와 실행 명령 |
| 서술: 라인 번호 참조 + 캐싱 설명 | 1.5점 | 반드시 본인 Dockerfile 라인 번호 참조 필수 |

**모범답안 Dockerfile:**

In [ ]:
%%writefile Dockerfile.backend
# Line 1: 베이스 이미지
FROM python:3.11-slim

# Line 3: curl 설치 (healthcheck용)
RUN apt-get update && apt-get install -y curl && rm -rf /var/lib/apt/lists/*

# Line 5: 작업 디렉토리 설정
WORKDIR /app

# Line 7-8: 의존성 파일 먼저 복사 (레이어 캐싱 최적화)
COPY pyproject.toml uv.lock ./

# Line 10: uv 설치 및 의존성 동기화
RUN pip install uv && uv sync

# Line 12-13: 나머지 소스 코드 복사
COPY . .

# Line 15: 포트 노출
EXPOSE 8000

# Line 17: 실행 명령
CMD ["uv", "run", "uvicorn", "app:app", "--host", "0.0.0.0", "--reload", "--port", "8000"]


**서술 모범답안 (의존성 선복사 이유):**

> "7~8번째 줄에서 `pyproject.toml`과 `uv.lock`을 먼저 복사하고, 10번째 줄에서 `uv sync`로 의존성을 설치합니다. 이렇게 하면 12~13번째 줄의 소스 코드만 변경했을 때, 10번째 줄의 의존성 설치 레이어가 Docker 캐시에서 재사용되어 빌드 시간이 크게 단축됩니다. 만약 `COPY . .`을 먼저 실행하면 소스 코드가 변경될 때마다 의존성 설치가 매번 다시 실행됩니다."

---

### 1-2. 고장난 docker-compose.yml 수정 (4점)

**채점 기준:** 각 오류 1점 (식별 0.5점 + 설명 0.5점)

**4개 오류 및 수정:**

**오류 1: `env_file` 경로**
- 잘못된 값: `./backend/.env.production`
- 올바른 값: `./backend/.env`
- 설명: 프로젝트에 `.env.production` 파일은 존재하지 않습니다.

**오류 2: healthcheck 포트**
- 잘못된 값: `http://localhost:3000/health`
- 올바른 값: `http://localhost:8000/health`
- 설명: backend 서비스는 포트 8000에서 실행됩니다.

**오류 3: backend 네트워크명**
- 잘못된 값: `app-network`
- 올바른 값: `agent-network`
- 설명: 하단에 `agent-network`만 정의되어 있습니다.

**오류 4: depends_on 형식**
- 잘못된 값: `depends_on: - backend` (단순 리스트)
- 올바른 값: `depends_on: backend: condition: service_healthy`
- 설명: `condition: service_healthy`를 사용해야 backend가 healthy 상태가 된 후 frontend가 시작됩니다.

**수정된 docker-compose.yml:**

In [ ]:
%%writefile docker-compose-fixed.yml
version: "3.8"
services:
  backend:
    build: ./backend
    ports:
      - "1234:8000"
    env_file:
      - ./backend/.env
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 40s
    networks:
      - agent-network

  frontend:
    build: ./frontend
    ports:
      - "1235:80"
    depends_on:
      backend:
        condition: service_healthy
    networks:
      - agent-network

networks:
  agent-network:
    driver: bridge


---

### 1-3. 환경변수 보안 사고 대응 (3점)

**(a) 3단계 긴급 대응 절차 (1.5점):**

1. **즉시 API 키 무효화 (1순위)**: OpenAI 대시보드에서 해당 키를 즉시 revoke/delete하고, 새 키를 생성합니다.
2. **git history에서 완전 제거**: `git filter-branch` 또는 `BFG Repo-Cleaner`를 사용하여 모든 커밋에서 `.env` 파일을 완전히 제거합니다.
3. **재발 방지 조치**: `.gitignore`에 `.env*` 패턴을 추가하고, `git-secrets`나 `pre-commit` hook을 설정합니다.

**채점 핵심**: API 키 무효화(revoke)가 반드시 1순위여야 합니다.

**(b) 답변 (1점):**

git은 모든 커밋의 전체 스냅샷을 저장합니다. 새 커밋으로 `.env` 파일을 삭제해도 이전 커밋 히스토리에 파일이 그대로 남아있습니다:

```bash
git log --all --full-history -- .env
git show <commit-hash>:.env
```

**(c) 완전 제거 도구 (0.5점):**

```bash
# BFG Repo-Cleaner
bfg --delete-files .env
git reflog expire --expire=now --all && git gc --prune=now --aggressive
git push --force

# 또는 git filter-repo
git filter-repo --path .env --invert-paths
```

---

## 문제 2: Nginx 리버스 프록시 및 멀티 컨테이너 (12점)

### 2-1. nginx.conf 직접 작성 (5점)

**채점 기준:**

| 항목 | 배점 |
|------|------|
| `events` + `http` + `server` 기본 구조 | 1점 |
| `proxy_pass http://backend:8000/;` (trailing slash) | 1점 |
| SSE 설정 3종 세트 | 2점 |
| `/health` 엔드포인트 | 0.5점 |
| `proxy_http_version 1.1` | 0.5점 |

**모범답안:**

In [ ]:
%%writefile nginx.conf
events {
    worker_connections 1024;
}

http {
    include /etc/nginx/mime.types;
    default_type application/octet-stream;

    gzip on;
    gzip_types text/plain text/css application/json application/javascript;

    server {
        listen 80;
        server_name localhost;

        # 정적 파일 서빙
        location / {
            root /usr/share/nginx/html;
            index index.html;
            try_files $uri $uri/ /index.html;
        }

        # 백엔드 API 프록시 (/api/ prefix 제거)
        location /api/ {
            proxy_pass http://backend:8000/;

            proxy_http_version 1.1;

            # SSE 스트리밍 필수 설정
            proxy_buffering off;
            proxy_cache off;
            chunked_transfer_encoding on;

            proxy_set_header Connection '';
        }

        # 헬스체크 엔드포인트
        location /health {
            access_log off;
            return 200 "OK\n";
            add_header Content-Type text/plain;
        }
    }
}


---

### 2-2. 컨테이너 네트워크 디버깅 (4점)

**(a) 502 원인 3가지 (확률 순):**

1. **nginx.conf의 proxy_pass 호스트명 오류**: `localhost` 대신 서비스명 `backend`를 사용해야 함
2. **backend와 frontend가 서로 다른 Docker 네트워크에 속함**
3. **backend 컨테이너가 아직 시작되지 않았거나 8000번 포트에서 리스닝하지 않음**

**(b) 디버깅 명령어:**

In [ ]:
# 원인 1 검증: nginx 설정에서 proxy_pass 호스트명 확인
!docker exec metacodelecture-agentservicedeploymentbasic-frontend-1 cat /etc/nginx/conf.d/default.conf

# 원인 2 검증: 네트워크 구성 및 소속 컨테이너 확인
!docker network inspect agent-network

# 원인 3 검증: backend 컨테이너 내부에서 health 엔드포인트 접근
!docker exec metacodelecture-agentservicedeploymentbasic-backend-1 curl -f http://localhost:8000/health


**(c) Docker 브릿지 네트워킹 서비스명 통신 원리:**

Docker Compose는 같은 네트워크 내의 컨테이너들에 대해 **내장 DNS 서버**(127.0.0.11)를 제공합니다. 서비스명(예: `backend`, `frontend`)을 DNS 호스트명으로 자동 등록하여, 같은 네트워크의 다른 컨테이너가 서비스명으로 요청하면 Docker 내장 DNS가 해당 컨테이너의 IP 주소로 해석합니다.

---

### 2-3. 헬스체크 설정 작성 (3점)

**(a) 헬스체크 설정:**

```yaml
healthcheck:
  test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
  interval: 30s
  timeout: 10s
  retries: 3
  start_period: 40s
```

**(b) 파라미터 역할:**

| 파라미터 | 설명 |
|----------|------|
| `interval` | 헬스체크 실행 간격. 30s면 30초마다 체크 |
| `timeout` | 단일 헬스체크의 응답 대기 시간. 10s 내 응답 없으면 실패 |
| `retries` | 연속 실패 허용 횟수. 3번 연속 실패 시 `unhealthy` |
| `start_period` | 컨테이너 시작 후 유예 기간. 이 기간 동안의 실패는 retries 카운트에 미포함 |

**(c) start_period=0 시나리오:**

1. **0초**: 컨테이너 시작. 즉시 첫 healthcheck 실행 -> 실패 (1회)
2. **30초**: 두 번째 healthcheck. 백엔드 아직 시작 중 -> 실패 (2회)
3. **60초**: 세 번째 healthcheck. 3회 연속 실패 -> `unhealthy` 전환
4. frontend의 `depends_on` + `service_healthy` 조건 미충족 -> **frontend 시작 안됨**
5. 백엔드는 실제로 정상이지만 이미 unhealthy로 마킹되어 전체 서비스 실패

---

## 문제 3: AWS ECS/Fargate 배포 아키텍처 (12점)

### 3-1. Dual ALB 아키텍처 다이어그램 (4점)

**텍스트 기반 모범답안:**

```
                        Internet
                       /        \
          +-----------+          +-----------+
          | Frontend  |          | Backend   |
          |   ALB     |          |   ALB     |
          | (port 80) |          | (port 80) |
          +-----------+          +-----------+
          [SG: 80,443            [SG: 80,443
           from 0.0.0.0/0]       from 0.0.0.0/0]
                |                      |
          +-----------+          +-----------+
          | frontend  |          | backend   |
          | target-gp |          | target-gp |
          | (port 80) |          | (port 8000)|
          +-----------+          +-----------+
                |                      |
          +-----------+          +-----------+
          | Frontend  |          | Backend   |
          | Fargate   |          | Fargate   |
          | (nginx:80)|---SSE--->| (FastAPI  |
          |           |          |  :8000)   |
          +-----------+          +-----------+
          [SG: 80 from           [SG: 8000 from
           frontend-alb-sg]       backend-alb-sg]

     Frontend nginx proxy_pass --> Backend ALB DNS
     SSE: Client <-- Frontend ALB <-- nginx <-- Backend ALB <-- FastAPI
```

---

### 3-2. 고장난 Task Definition 수정 (4점)

**3개 오류:**

**오류 1: `executionRoleArn` 역할 이름 (1점)**
- 잘못된 값: `wrongRoleName`
- 올바른 값: `ecsTaskExecutionRole-dev`

**오류 2: `secrets` 블록 형식 (1점)**
- 잘못된 값: `{"name": "OPENAI_API_KEY", "name": "openai-api-key-dev"}` (중복 key)
- 올바른 값: `{"name": "OPENAI_API_KEY", "valueFrom": "arn:aws:secretsmanager:...:secret:dev/openai-api-key"}`

**오류 3: `awslogs-group` 이름 (1점)**
- 잘못된 값: `/ecs/wrong-log-group`
- 올바른 값: `/ecs/backend-dev`

**executionRoleArn vs taskRoleArn 차이 (1점):**

`executionRoleArn`은 **ECS 에이전트**가 컨테이너를 시작하기 위해 사용하는 IAM 역할입니다 (ECR pull, CloudWatch Logs, Secrets Manager 조회). `taskRoleArn`은 **컨테이너 내부의 애플리케이션 코드**가 AWS 서비스에 접근할 때 사용합니다 (S3, DynamoDB 등). 분리하면 최소 권한 원칙을 적용할 수 있습니다.

---

### 3-3. ECS 배포 환경에서의 Health Check 설계 (4점)

**(a) Health Check 3단계 실패 시 동작 (2점)**

| Health Check 레벨 | 설정 위치 | 실패 시 동작 | 확인 방법 |
|---|---|---|---|
| 컨테이너 | Task Definition `healthCheck` | ECS가 Task를 `UNHEALTHY`로 마킹 → Task 중지 → 새 Task 시작 (서비스의 desired count 유지) | `aws ecs describe-tasks --tasks <task-id>` → `healthStatus` 필드 |
| ALB Target Group | Target Group health check | Target을 `unhealthy`로 마킹 → ALB가 해당 Target으로 트래픽 전송 중단 → 다른 healthy Target으로 라우팅 | AWS Console > EC2 > Target Groups > 해당 TG > Targets 탭 |
| 애플리케이션 | FastAPI `/health`, `/ready` | `/health` 실패: 위 두 레벨의 health check가 실패로 판정됨. `/ready` 실패: LLM/Pinecone 미초기화 상태로, 요청 처리는 가능하나 일부 기능(RAG, Agent) 불가 | `curl http://<alb-dns>/api/ready` |

**채점**: 3개 레벨 모두 정확히 설명 시 2점. 1개 누락 시 1점. 2개 이상 누락 시 0점.


In [ ]:
# (b) ALB Target Group Health Check 모범답안 (1점)

hcl_answer = """
resource "aws_lb_target_group" "backend" {
  name        = "backend-tg-${var.environment}"
  port        = 8000
  protocol    = "HTTP"
  vpc_id      = var.vpc_id
  target_type = "ip"

  health_check {
    path                = "/health"
    port                = "8000"
    protocol            = "HTTP"
    healthy_threshold   = 3
    unhealthy_threshold = 3
    interval            = 30
    timeout             = 5
  }
}
"""
print(hcl_answer)
# 채점: path=/health, port=8000 필수. threshold/interval/timeout 합리적 값이면 정답.


**(c) `/health` vs `/ready` 차이 모범답안 (1점)**

- **`/health`** (Liveness): 서버 프로세스가 살아있는지만 확인. `{"status": "healthy"}`를 항상 반환. LLM/Pinecone 연결 여부와 무관. ALB/컨테이너 health check에 사용.
- **`/ready`** (Readiness): LLM과 Pinecone이 모두 초기화되었는지 확인. `{"status": "ready", "checks": {"llm": true, "vectorstore": true}}`를 반환. 하나라도 false이면 `not_ready`.
- **용도 차이**: `/health`가 실패하면 컨테이너 자체에 문제가 있으므로 재시작해야 함 (Liveness). `/ready`가 실패하면 컨테이너는 살아있지만 요청을 처리할 준비가 안 된 상태 (Readiness). Kubernetes의 livenessProbe/readinessProbe와 동일한 개념.

**채점**: `/health`=프로세스 존재 확인, `/ready`=의존성 초기화 확인, Liveness/Readiness 구분 3가지 모두 언급 시 1점.


---


---

## 문제 4: GCP Cloud Run 배포 (10점)

### 4-1. AWS vs GCP 배포 비교 (4점)

**모범답안 비교표:**

| 비교 항목 | AWS ECS/Fargate | GCP Cloud Run | 더 간단한 쪽 + 이유 |
|-----------|----------------|---------------|-------------------|
| 이미지 레지스트리 | ECR (`aws ecr get-login-password ...`) | Artifact Registry (`gcloud auth configure-docker ...`) | **GCP** -- 명령어 1줄 인증 |
| 컨테이너 오케스트레이션 | ECS Cluster + Service + Task Definition (3단계) | `gcloud run deploy` 한 줄 | **GCP** -- 별도 클러스터 불필요 |
| 로드밸런싱 | ALB 수동 생성 + Target Group | 자동 제공 (HTTPS LB 기본 포함) | **GCP** -- 설정 불필요 |
| 시크릿 관리 | Secrets Manager + IAM + Task Definition secrets | `gcloud run deploy --set-secrets=...` | **GCP** -- 플래그 하나로 주입 |
| 로깅 | CloudWatch Logs (logConfiguration 필수) | Cloud Logging (stdout 자동 수집) | **GCP** -- 설정 없이 자동 수집 |
| 오토스케일링 | Auto Scaling Target + Policy 별도 설정 | `--min-instances=0 --max-instances=10` 플래그 | **GCP** -- 배포 명령어에 포함 |
| Cold Start | 없음 (항상 최소 1개 Task) | 있음 (`min-instances=0`일 때) | **AWS** -- Cold Start 없음 |
| 비용 모델 | 시간당 과금 (항상 실행) | 요청당 과금 (미사용 시 무료) | **상황에 따라** -- 트래픽 적으면 GCP |

---

### 4-2. Cloud Run용 nginx.conf 수정 (3점)

**모범답안:**

In [ ]:
%%writefile nginx-cloudrun.conf
location /api/ {
    # Cloud Run은 HTTPS만 허용
    proxy_pass https://backend-dev-xxxxx-an.a.run.app/;

    proxy_http_version 1.1;

    # Cloud Run HTTPS 연결에 필수
    proxy_ssl_server_name on;

    # Cloud Run은 Host 헤더로 서비스를 라우팅
    proxy_set_header Host backend-dev-xxxxx-an.a.run.app;

    # SSE 스트리밍 설정
    proxy_buffering off;
    proxy_cache off;
    proxy_read_timeout 300s;
    chunked_transfer_encoding on;
    proxy_set_header Connection '';
}


**3가지 차이점:**

**(a) HTTPS vs HTTP:** Cloud Run은 외부 트래픽에 HTTPS를 강제. `proxy_pass https://...`로 변경 필요.

**(b) Host 헤더:** Cloud Run은 Host 헤더로 서비스를 라우팅. 없으면 404.

**(c) proxy_read_timeout:** 인터넷 경유 + Cold Start로 지연 발생. SSE 장시간 연결을 위해 300s 이상 필요.

---

### 4-3. Cold Start 분석 (3점)

**(a) 초기화 프로세스:**
1. Cloud Run이 새 컨테이너 인스턴스 프로비저닝
2. Artifact Registry에서 이미지 다운로드
3. 컨테이너 시작 및 Linux 커널 초기화
4. Python 인터프리터 로드
5. uv 가상 환경 활성화 및 패키지 임포트
6. FastAPI 앱 초기화
7. `init_llm()`: OpenAI 클라이언트 초기화
8. `init_pinecone()`: PineconeVectorStore 초기화
9. 첫 번째 요청 처리 시작

**(b) Cold Start 감소 전략:**

| 전략 | 효과 | 트레이드오프 |
|------|------|-------------|
| `--min-instances=1` | Cold Start 완전 제거 | 비용 증가 ~$15-30/월 |
| `--cpu-boost` | 초기화 시간 ~30% 감소 | 효과 제한적 |
| 이미지 크기 최적화 | 다운로드 시간 감소 | 개발 복잡도 증가 |

**(c) min-instances=1 선택 시점:** 사용자 대면 프로덕션 서비스에서 SLA 응답 시간 보장이 필요할 때, 실시간 챗봇 서비스, webhook 수신 서비스 등.

---

## 문제 5: Terraform Infrastructure as Code (14점)

### 5-1. Terraform Security Group 리소스 작성 (5점)

**모범답안:**

In [ ]:
%%writefile security-groups.tf
# Backend ALB Security Group
resource "aws_security_group" "backend_alb" {
  name        = "backend-alb-sg-${var.environment}"
  description = "Security group for Backend ALB"
  vpc_id      = var.vpc_id

  ingress {
    description = "HTTP from internet"
    from_port   = 80
    to_port     = 80
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    description = "HTTPS from internet"
    from_port   = 443
    to_port     = 443
    protocol    = "tcp"
    cidr_blocks = ["0.0.0.0/0"]
  }

  egress {
    description = "All outbound traffic"
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }

  tags = {
    Name        = "backend-alb-sg-${var.environment}"
    Environment = var.environment
  }
}

# Backend Container Security Group
resource "aws_security_group" "backend" {
  name        = "backend-sg-${var.environment}"
  description = "Security group for backend ECS tasks"
  vpc_id      = var.vpc_id

  ingress {
    description     = "Backend port from Backend ALB"
    from_port       = 8000
    to_port         = 8000
    protocol        = "tcp"
    security_groups = [aws_security_group.backend_alb.id]
  }

  egress {
    description = "All outbound traffic"
    from_port   = 0
    to_port     = 0
    protocol    = "-1"
    cidr_blocks = ["0.0.0.0/0"]
  }

  tags = {
    Name        = "backend-sg-${var.environment}"
    Environment = var.environment
  }
}


**서술:** `security_groups`로 ALB SG를 참조하면, ALB를 거치지 않는 직접 접근이 원천 차단됩니다. `cidr_blocks = ["0.0.0.0/0"]`으로 설정하면 누구나 컨테이너의 8000번 포트에 직접 접근할 수 있어 ALB를 우회한 공격이 가능합니다.

---

### 5-2. Terraform 변수 및 출력 정의 (4점)

**variables.tf 모범답안:**

In [ ]:
%%writefile variables.tf
variable "environment" {
  description = "Deployment environment (dev, stag, prod)"
  type        = string
  default     = "dev"

  validation {
    condition     = contains(["dev", "stag", "prod"], var.environment)
    error_message = "Environment must be one of: dev, stag, prod."
  }
}

variable "vpc_id" {
  description = "VPC ID (no default - must be provided per environment)"
  type        = string
}

variable "public_subnet_ids" {
  description = "Public subnet IDs for ALB (minimum 2 AZs required)"
  type        = list(string)

  validation {
    condition     = length(var.public_subnet_ids) >= 2
    error_message = "At least 2 public subnets required."
  }
}

variable "backend_cpu" {
  description = "Backend task CPU units"
  type        = string
  default     = "512"
}

variable "backend_memory" {
  description = "Backend task memory in MiB"
  type        = string
  default     = "1024"
}

variable "backend_desired_count" {
  description = "Number of backend tasks to run"
  type        = number
  default     = 2
}


In [ ]:
%%writefile outputs.tf
output "frontend_alb_dns" {
  description = "Frontend ALB DNS name"
  value       = aws_lb.frontend.dns_name
}

output "backend_alb_dns" {
  description = "Backend ALB DNS name"
  value       = aws_lb.backend.dns_name
}

output "ecs_cluster_name" {
  description = "ECS cluster name"
  value       = aws_ecs_cluster.main.name
}


**vpc_id에 default가 없어야 하는 이유:** VPC ID는 계정/리전/환경마다 고유한 값. 하드코딩하면 다른 환경에서 재사용 불가. `terraform.tfvars`로 반드시 외부에서 주입해야 합니다.

---

### 5-3. 수동 배포 vs Terraform 비교 (5점)

**(a) 수동 재구성 문제점:** 1) 구성 누락 위험 2) 환경 불일치 (Configuration Drift) 3) 시간 소요 및 휴먼 에러

**(b) Terraform 해결:** 1) 코드로 정의되어 누락 없음 2) 동일 코드로 일관성 보장 3) `terraform apply`로 자동화

**(c) 4개 Terraform 명령어:**

In [ ]:
# 1. 초기화
# terraform init -backend-config="bucket=metacode-terraform-state" \
#                -backend-config="key=agent-service/dr/terraform.tfstate" \
#                -backend-config="region=ap-southeast-1"

# 2. 실행 계획 확인
# terraform plan -var-file="dr.tfvars"

# 3. 인프라 생성
# terraform apply -var-file="dr.tfvars"

# 4. 생성된 리소스 확인
# terraform output

# 순서 이유: init -> plan (검증) -> apply (생성) -> output (확인)


In [ ]:
%%writefile terraform.tfvars
environment           = "prod"
image_tag             = "v1.0.0"
vpc_id                = "vpc-0123456789abcdef0"
public_subnet_ids     = ["subnet-aaa111", "subnet-bbb222"]
backend_cpu           = "1024"
backend_memory        = "2048"
backend_desired_count = 3
frontend_cpu          = "512"
frontend_memory       = "1024"
frontend_desired_count = 2
# 유효성: backend 1024 CPU + 2048 Memory (OK), frontend 512 CPU + 1024 Memory (OK)


---

## 문제 6: CI/CD with GitHub Actions (12점)

### 6-1. 고장난 GitHub Actions 워크플로우 수정 (5점)

**오류 1:** `branches: [main, develop]` -> `[main, develop, staging]` (staging 누락)

**오류 2:** `main && 'development'` -> `main && 'production'` (매핑 반대)

**오류 3:** `terraform init`에 key와 region 누락 -> `-backend-config="key=..."` `-backend-config="region=..."` 추가

**오류 4:** `docker build -f ./Dockerfile` -> `-f ./backend/Dockerfile` (Dockerfile 경로 수정)

**3-Stage 배포 전략:** ECR 리포지토리가 존재해야 이미지를 push할 수 있고, ECR에 이미지가 있어야 ECS Task Definition이 이미지를 참조하여 컨테이너를 실행할 수 있습니다.

---

### 6-2. 브랜치 전략 및 환경 매핑 (3점)

**다이어그램:**
```
feature/login --PR--> develop --PR--> staging --PR--> main
                         |                |              |
                    [auto deploy]    [auto deploy]  [auto deploy]
                         |                |              |
                    development        staging       production
                                                   (수동 승인 필요)
```

**이벤트 시퀀스:** feature 작업 -> develop PR + merge -> dev 자동 배포 -> staging PR + merge -> staging 자동 배포 -> main PR + merge -> Environment Protection Rules 수동 승인 -> production 배포

---

### 6-3. 배포 스텝 YAML 작성 (4점)

**모범답안:**

In [ ]:
%%writefile ecr-push-steps.yml
- name: Login to Amazon ECR
  id: ecr-login
  uses: aws-actions/amazon-ecr-login@v2

- name: Build and Push Backend Image
  env:
    ECR_REGISTRY: ${{ steps.ecr-login.outputs.registry }}
    IMAGE_TAG: backend-${{ vars.ENVIRONMENT }}:v${{ vars.IMAGE_TAG }}
  run: |
    docker build -t $ECR_REGISTRY/$IMAGE_TAG -f ./backend/Dockerfile ./backend
    docker push $ECR_REGISTRY/$IMAGE_TAG

- name: Build and Push Frontend Image
  env:
    ECR_REGISTRY: ${{ steps.ecr-login.outputs.registry }}
    IMAGE_TAG: frontend-${{ vars.ENVIRONMENT }}:v${{ vars.IMAGE_TAG }}
  run: |
    docker build -t $ECR_REGISTRY/$IMAGE_TAG -f ./frontend/Dockerfile ./frontend
    docker push $ECR_REGISTRY/$IMAGE_TAG


---

## 문제 7: 멀티 클라우드 아키텍처 이해 (8점)

### 7-1. 아키텍처 의사결정 문서 (4점)

**(a) 추천: GCP Cloud Run** -- 2인 스타트업 + 일 100건 + $200 예산에 최적

**(b) 리소스 구성:** Backend: 1 vCPU, 512 MiB, min=0, max=3 / Frontend: 1 vCPU, 256 MiB, min=0, max=2

**(c) 비용 계산:**

| 시나리오 | Cloud Run | AWS Fargate |
|----------|-----------|-------------|
| 일 100건 (현재) | ~$2-3/월 | ~$59/월 |
| 일 10,000건 (6개월 후) | ~$55-80/월 | ~$70-100/월 |

**(d) 마이그레이션 플랜:**
- 단기(~1,000건): min-instances=1 설정
- 중기(~5,000건): max-instances=5 확장
- 장기(~10,000건+): max-instances=10, 필요 시 AWS ECS 마이그레이션 검토

---

### 7-2. 크로스 클라우드 시크릿 관리 (4점)

**Part A -- AWS ECS 시크릿 Terraform:**

In [ ]:
%%writefile secrets.tf
# Secrets Manager에서 시크릿 ARN 조회
data "aws_secretsmanager_secret" "openai" {
  name = var.openai_secret_name
}

# ECS Task Definition 내 secrets 블록
resource "aws_ecs_task_definition" "backend" {
  # ... 기존 설정 ...
  container_definitions = jsonencode([{
    name  = "backend"
    image = "${aws_ecr_repository.backend.repository_url}:${var.image_tag}"
    secrets = [
      {
        name      = "OPENAI_API_KEY"
        valueFrom = data.aws_secretsmanager_secret.openai.arn
      }
    ]
  }])
}


In [ ]:
# Part B: GCP Cloud Run 시크릿 명령어
gcloud_cmd = """
gcloud run deploy backend-dev \\
  --image asia-northeast3-docker.pkg.dev/PROJECT_ID/agent-tf/backend:latest \\
  --region asia-northeast3 \\
  --set-secrets=OPENAI_API_KEY=openai-api-key:latest \\
  --set-secrets=PINECONE_API_KEY=pinecone-api-key:latest
"""
print(gcloud_cmd)


**Part C -- WIF 안전한 이유:**
1. **Short-lived token**: 단기 토큰(기본 1시간) 사용, 유출 시 자동 만료
2. **키 파일 불필요**: OIDC 토큰 교환 방식, 파일 유출 위험 없음
3. **자동 갱신**: 수동 로테이션 불필요, 보안 위험 누적 방지

---

## 문제 8: 프로덕션 운영 개념 (12점)

### 8-1. OOM Kill 원인 및 해결 (4점)

**(a) 기술 스택 특화 OOM 원인:**
1. **LangChain 임베딩 모델 메모리 누적**: OpenAIEmbeddings + PineconeVectorStore similarity_search 벡터 결과 누적
2. **동시 SSE 스트림 버퍼 누적**: StreamingResponse AsyncGenerator 각 연결마다 메모리 사용
3. **RAG 컨텍스트 메모리 사용**: 검색된 문서 전체 내용을 문자열로 결합, 동시 요청 시 누적
4. **Python GC 지연 및 LLM 응답 토큰 누적**: 대형 객체 즉시 해제 안됨

**(c) 해결 방안:**
- 단기: 메모리 증가 1024->2048, 동시 연결 수 제한
- 중기: RAG 컨텍스트 크기 제한, 메모리 프로파일링
- 장기: 수평 스케일링, SSE 분리 아키텍처

**(d) Terraform:** `backend_memory = "2048"` (512 CPU에서 호환 가능 범위 1024~4096)

In [ ]:
# (b) 진단 명령어

# 1. Container Insights 메모리 모니터링
# aws cloudwatch get-metric-data --metric-data-queries '[{"Id":"mem","MetricStat":{...}}]'

# 2. ECS Task 중지 사유 확인
# aws ecs describe-tasks --cluster agent-cluster-dev --tasks <task-id> \
#   --query 'tasks[0].{stoppedReason:stoppedReason,stopCode:stopCode}'

# 3. 애플리케이션 수준 메모리 추적
import tracemalloc
tracemalloc.start()
snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics('lineno')
for stat in top_stats[:10]:
    print(stat)

# 4. 동시 연결 수 모니터링
# aws logs filter-log-events --log-group-name /ecs/backend-dev \
#   --filter-pattern '"[/ask] Question"'


---

### 8-2. 503 에러 디버깅 방법론 (4점)

**(a) 첫 5분 확인 사항:**
1. ECS Service 이벤트 확인 (0~1분)
2. Target Group 상태 확인 (1~2분)
3. ECS Task 로그 확인 (2~3분)
4. ALB Access Log 확인 (3~4분)
5. 리소스 사용량 확인 (4~5분)

**(b) Target 상태별 대응:**

| Target 상태 | 의미 | 대응 |
|-------------|------|------|
| `unhealthy` | Health check 연속 실패 | Task 로그/SG 규칙 확인 |
| `draining` | Task 종료 중 | 정상 동작, 배포 중이면 대기 |
| `initial` | 첫 health check 대기 | 정상, interval 경과 대기 |
| `unused` | AZ 미활성화 | 서브넷 구성 확인 |

**(d) 의사결정 트리:**
```
503 에러
├── Running Task 있는가?
│   ├── NO -> Task Definition 오류? / 리소스 부족?
│   └── YES -> Target Group 상태 확인
├── healthy 타겟 있는가?
│   ├── NO -> health check 실패 (앱/SG/포트 확인)
│   └── YES -> ALB 설정 확인
├── ALB Listener/Rule 올바른가?
└── 네트워크 문제? (IGW, Route Table, NACL)
```

In [ ]:
# (c) AWS CLI 명령어

# 1. ECS Service 이벤트
# aws ecs describe-services --cluster agent-cluster-dev --services backend-dev-service --query 'services[0].events[:10]'

# 2. Target Group 상태
# aws elbv2 describe-target-health --target-group-arn <backend-tg-arn>

# 3. 실행 중인 Task 확인
# aws ecs list-tasks --cluster agent-cluster-dev --service-name backend-dev-service
# aws ecs describe-tasks --cluster agent-cluster-dev --tasks <task-id>

# 4. CloudWatch 로그 실시간 확인
# aws logs tail /ecs/backend-dev --follow --since 5m


---

### 8-3. 배포 전략 및 Circuit Breaker (4점)

**Rolling Update:**
```
Task v1: [████████]--------
Task v2:     [----████████]
```
적합: 점진적 교체, 제로 다운타임. 비용 오버헤드: 낮음

**Blue-Green:**
```
Blue(v1):  [████████████]
Green(v2):     [████████████]
트래픽:    [--Blue--][Green-]
```
적합: 즉각 롤백 필요. 비용 오버헤드: 높음 (2배 리소스)

**Canary:**
```
v1(95%): [████████████████]
v2(5%):  [████] -> [25%] -> [100%]
```
적합: 대규모 사용자 기반에서 점진적 검증. 비용 오버헤드: 중간

**Circuit Breaker 없이 배포 실패 시:** Task 시작 -> health check 실패 -> ECS가 재시작 -> 같은 이유로 실패 -> 무한 반복 -> 모든 v1도 교체되며 서비스 전체 다운

In [ ]:
%%writefile circuit-breaker.tf
resource "aws_ecs_service" "backend" {
  name            = "backend-${var.environment}-service"
  cluster         = aws_ecs_cluster.main.id
  task_definition = aws_ecs_task_definition.backend.arn
  desired_count   = var.backend_desired_count
  launch_type     = "FARGATE"

  deployment_configuration {
    maximum_percent         = 200
    minimum_healthy_percent = 50

    deployment_circuit_breaker {
      enable   = true
      rollback = true
    }
  }
}


---

## 문제 9: 구조화 로깅 및 비용 최적화 (8점)

### 9-1. 구조화 로깅 구현 (4점)

**Part A -- Python 로깅 설정:**

In [ ]:
import logging

class TraceIdFormatter(logging.Formatter):
    """trace_id가 없는 로그 레코드에 기본값 'N/A'를 제공"""
    def format(self, record):
        if not hasattr(record, 'trace_id'):
            record.trace_id = 'N/A'
        return super().format(record)

# 로거 설정
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# 핸들러 + JSON 포맷터
handler = logging.StreamHandler()
handler.setFormatter(TraceIdFormatter(
    '{"time": "%(asctime)s", "level": "%(levelname)s", '
    '"trace_id": "%(trace_id)s", "message": "%(message)s"}'
))
logger.addHandler(handler)

# 테스트
logger.info("Request received", extra={"trace_id": "abc-123"})
logger.info("Application starting")  # trace_id -> 'N/A'


**Part B -- Docker 멀티스테이지 빌드 (1.5점)**

기존 단일 스테이지 Dockerfile을 멀티스테이지로 최적화한 모범답안:


In [ ]:
# Part B 모범답안: 멀티스테이지 Dockerfile

dockerfile_multistage = """
# Stage 1: Builder
FROM python:3.11-slim AS builder

WORKDIR /app

COPY pyproject.toml uv.lock ./

RUN pip install uv && uv sync

COPY . .

# Stage 2: Runtime
FROM python:3.11-slim

RUN apt-get update && apt-get install -y curl && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# builder에서 .venv와 소스만 복사
COPY --from=builder /app/.venv ./.venv
COPY --from=builder /app .

EXPOSE 8000

# production에서는 --reload 제거
CMD [".venv/bin/uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""
print(dockerfile_multistage)

# 채점 기준:
# - 2개 스테이지(builder + runtime) 구분: 0.5점
# - COPY --from=builder 사용: 0.5점
# - --reload 제거: 0.5점
# 이미지 크기 비교 스크린샷은 가점(필수 아님)


**채점 포인트:**
- 2개 스테이지(builder + runtime) 구분: 0.5점
- `COPY --from=builder` 사용: 0.5점
- `--reload` 제거 (production 적합): 0.5점

**참고**: 단일 스테이지 이미지 ~800MB → 멀티스테이지 ~300MB 수준으로 감소 기대


---


---

### 9-2. 비용 최적화 전략 (4점)

| 전략 | 변경 내용 | 예상 절감 |
|------|----------|----------|
| 1. Fargate Spot (dev/stag) | dev/stag capacity provider를 FARGATE_SPOT으로 변경 | ~$70/월 |
| 2. 로그 보존 기간 축소 | dev: 7일, stag: 14일, prod: 30일 | ~$48/월 |
| 3. ECR 라이프사이클 정책 | 최근 5개 이미지만 유지 | ~$16/월 |
| 4. Container Insights 비활성화 (dev/stag) | dev/stag에서 비활성화 | ~$6/월 |
| 5. dev 환경 스케줄 삭제 | 업무 외 시간 desired_count=0 | ~$30/월 |

**총 예상 절감: ~$170/월 -> 목표 $300/월 달성**

In [ ]:
%%writefile cost-optimization.tf
# 전략 1: Fargate Spot
resource "aws_ecs_cluster_capacity_providers" "main" {
  cluster_name = aws_ecs_cluster.main.name
  capacity_providers = ["FARGATE", "FARGATE_SPOT"]

  default_capacity_provider_strategy {
    base              = 0
    capacity_provider = var.environment == "prod" ? "FARGATE" : "FARGATE_SPOT"
    weight            = 1
  }
}

# 전략 2: 로그 보존 기간 축소
resource "aws_cloudwatch_log_group" "backend" {
  name              = "/ecs/backend-${var.environment}"
  retention_in_days = var.environment == "prod" ? 30 : 7

  tags = {
    Name        = "/ecs/backend-${var.environment}"
    Environment = var.environment
  }
}


---

## 채점 요약표

| 문제 | 배점 | 0점 조건 요약 |
|------|------|--------------|
| 1-1. Dockerfile | 5점 | 빌드 실패 / 라인 번호 미참조 |
| 1-2. docker-compose 수정 | 4점 | 3개 미만 수정 |
| 1-3. 보안 사고 대응 | 3점 | API 키 무효화 1순위 아님 |
| 2-1. nginx.conf | 5점 | SSE 설정 누락 / prefix 미제거 |
| 2-2. 네트워크 디버깅 | 4점 | DNS 미언급 / 프로젝트 특정 이름 미사용 |
| 2-3. 헬스체크 | 3점 | frontend 시작 영향 미설명 |
| 3-1. 아키텍처 다이어그램 | 4점 | SG 세부사항 누락 / 배포 증거 없음 |
| 3-2. Task Definition 수정 | 4점 | executionRole vs taskRole 미설명 |
| 3-3. Health Check 설계 | 4점 | 문법 오류 / 스크린샷 없음 |
| 4-1. AWS vs GCP 비교 | 4점 | 6개 미만 항목 / CLI 없음 |
| 4-2. Cloud Run nginx.conf | 3점 | Host 헤더 미설정 |
| 4-3. Cold Start 분석 | 3점 | 초기화 프로세스 미언급 |
| 5-1. Security Group HCL | 5점 | cidr_blocks 사용 / 서술 없음 |
| 5-2. variables + outputs | 4점 | validation 없음 / 변수 4개 미만 |
| 5-3. 수동 vs Terraform | 5점 | 무효 CPU/Memory 조합 |
| 6-1. GitHub Actions 수정 | 5점 | 3개 미만 수정 |
| 6-2. 브랜치 전략 | 3점 | PR 미언급 / staging 생략 |
| 6-3. 배포 스텝 YAML | 4점 | ECR 로그인 누락 |
| 7-1. 아키텍처 ADR | 4점 | 비용 계산 없음 |
| 7-2. 시크릿 관리 | 4점 | value/valueFrom 혼동 |
| 8-1. OOM Kill | 4점 | 원인이 일반적 / 진단 명령 없음 |
| 8-2. 503 디버깅 | 4점 | CLI 3개 미만 / 의사결정 트리 없음 |
| 8-3. Circuit Breaker | 4점 | HCL 불가 / 다이어그램 없음 |
| 9-1. 구조화 로깅 | 4점 | trace_id 처리 누락 |
| 9-2. 비용 최적화 | 4점 | 4개 미만 전략 / HCL 없음 |
| **합계** | **100점** | |